## WCUS Trips cleaning steps
Get the data [here](https://ndclibrary.sec.usace.army.mil/resource?title=2023%20WCUS%20Trips%20-%20All%20Regions%20&documentId=07ddf14b-1522-4c6f-d894-40cd74d58a7f).

1. Filter to get only In/Out/Thru=="Outbound Shipping" or "Inbound Shipping"
2. Filter to get only TrafficCode==1 (aka TrafficName=="Domestic (Trips & Drafts)")
3. Keep only the columns:
    * 'RegionName', 'Up/Down', 'VesselType', 'VesselTypeName', 'VesselDraftFt', 'Trips', 'CompletedYear'
4. Create a few separate data sets: 
    * Filter to get only WaterwayCode==3924 (aka WaterwayName=="Duluth-Superior, MN and WI")
    * Filter to get only RegionName=="GREAT LAKES"
5. Save as something descriptive like "WCUS_Trips_DuluthSuperior_Outbound_Domestic_2014-2023.csv"

## List of Top Iron Ore Outports and Inports

In [5]:
import pandas as pd

In [241]:
# No Thunder Bay. That is in ON.
# Taconite and Silver Bay (last two) are not USACE ports

# Careful! Cities and their harbors are listed separately
# E.g. 3841 for Marquette Harbor (admin, maintenance) vs. 3844 for Marquette Township ()
# 3619 is the Presque Isle farther east, by Alpena

outports = pd.DataFrame({
    'WaterwayCode':[3924, 3926, 3845, 3841]#, 3929, 3928]
    })



In [242]:
# Note also:
# Cleveland-Cliffs Dearborn Works in Michigan
# Great Lakes Steel in Michigan
# These facilities ultimately receive taconite by rail, though

# Make sure to distinguish between receipts and throughports
    # E.g. Chicago is just a rail hub, other places go straight to mills nearby
# Calumet and Chicago included at end just because Calumet is high-tonnage and Chicago is famous
# Not much iron ore though
inports = pd.DataFrame({
    'WaterwayCode': [3738, 3736, 3739, 3204, 3217, 3315, 3220, 3219, 3741, 3747]
    })

## WCUS Trips Cleaning

In [23]:
path = "../data/raw/Trips_AllRegions_10yr_2014-2023.xlsx"

wcus = pd.read_excel(path, sheet_name="Trips_AllRegions_10yr_2014-2023")

In [24]:
# pd.set_option('display.max_rows', None)
# wcus.loc[:, ['WaterwayCode', 'WaterwayName']].drop_duplicates()

In [243]:
wcus.merge(outports, how='right').loc[:, ['WaterwayName', 'WaterwayCode']].drop_duplicates()
# wcus.loc[wcus['WaterwayCode'].isin(outports['WaterwayCode']), 'WaterwayName'].unique()

,WaterwayName,WaterwayCode
0,"Duluth-Superior, MN and WI",3924
1152,"Two Harbors, MN",3926
1752,"Presque Isle, MI",3845
2237,"Marquette, MI",3841


In [244]:
# wcus.merge(inports, how='right')['WaterwayName'].unique()
wcus.loc[wcus['WaterwayCode'].isin(inports['WaterwayCode']), 'WaterwayName'].unique()

array(['Ashtabula Port Authority, OH', 'Burns Waterway Harbor, IN',
       'Calumet Harbor and River, IL and IN', 'Chicago Harbor, IL',
       'Cleveland-Cuyahoga Port, OH', 'Conneaut Harbor, OH', 'Gary, IN',
       'Indiana Harbor, IN', 'Rouge River, MI',
       'Toledo-Lucas County Port, OH'], dtype=object)

In [246]:
# TODO
# Now filter into chart-ready CSVs
    # At least 12ft draft when in ballast for ore-carrying vessels according to 2020 LCA Annual Report, Fleet Profile section
    # Up/Down is apparently nonsense, disregard
    # Just Duluth-Superior
        # One csv for Dry Cargo Barge
        # One csv for Self-Propelled dry
# Add up trips if multiple ports belong to a USACE Port Statistical Area
min_draft_ft = 17

wcus_all_out = wcus.loc[
        wcus['WaterwayCode'].isin(outports['WaterwayCode']) 
            & wcus['VesselTypeName'].isin(['Self-Propelled dry'])
            & (wcus['In/Out/Thru'] == 'Outbound Shipping')
            & (wcus['VesselDraftFt'] >= min_draft_ft)
            & (wcus['TrafficCode'] == 1) # Domestic (Trips & Drafts)
            ,
        :
    ]
wcus_all_out.head(3)

,RegionName,WaterwayCode,WaterwayName,TrafficCode,TrafficName,In/Out/Thru,Up/Down,VesselType,VesselTypeName,VesselDraftFt,Trips,CompletedYear
149502,GREAT LAKES,3924,"Duluth-Superior, MN and WI",1,Domestic (Trips & Drafts),Outbound Shipping,Port,1.0,Self-Propelled dry,17,10,2014
149503,GREAT LAKES,3924,"Duluth-Superior, MN and WI",1,Domestic (Trips & Drafts),Outbound Shipping,Port,1.0,Self-Propelled dry,17,9,2015
149504,GREAT LAKES,3924,"Duluth-Superior, MN and WI",1,Domestic (Trips & Drafts),Outbound Shipping,Port,1.0,Self-Propelled dry,17,4,2016


In [247]:
cols_keep = ['WaterwayCode', 'WaterwayName', 'In/Out/Thru', 'VesselTypeName', 'VesselDraftFt', 'Trips', 'CompletedYear']



### Duluth-Superior Outbound Trip Counts (17ft+ draft depth)

In [248]:
trips_duluth = wcus_all_out.loc[
    wcus_all_out['WaterwayName'] == 'Duluth-Superior, MN and WI',
    cols_keep
]

In [249]:
trip_counts_duluth = trips_duluth.groupby(['WaterwayCode', 'WaterwayName', 'In/Out/Thru', 'VesselTypeName', 'CompletedYear']).sum().reset_index()


In [250]:
# trip_counts_duluth.drop(columns=['VesselDraftFt']).to_csv('../data/clean/trip_counts/outbound/duluth.csv')

### All outport trips

In [251]:
wcus.merge(outports, how='right').loc[:, ['WaterwayName', 'WaterwayCode']].drop_duplicates()


,WaterwayName,WaterwayCode
0,"Duluth-Superior, MN and WI",3924
1152,"Two Harbors, MN",3926
1752,"Presque Isle, MI",3845
2237,"Marquette, MI",3841


In [252]:
outports_dict = dict(zip(outports['WaterwayCode'], ['duluth_superior', 'two_harbors', 'presque_isle', 'marquette_harbor']))


In [253]:
for code, port in outports_dict.items():
    trips_port = wcus_all_out.loc[
        wcus_all_out['WaterwayCode'] == code,
        cols_keep
    ]
    trip_counts_port = trips_port.groupby(['WaterwayCode', 'WaterwayName', 'In/Out/Thru', 'VesselTypeName', 'CompletedYear']).sum().reset_index()

    trip_counts_port.drop(columns=['VesselDraftFt']).to_csv(
            '../data/clean/trip_counts/outbound/{}.csv'.format(port),
            index=False
        )


### Burns Inbound Trip Counts ("")

In [170]:
wcus_all_in = wcus.loc[
        wcus['WaterwayCode'].isin(inports['WaterwayCode']) 
            & wcus['VesselTypeName'].isin(['Self-Propelled dry'])
            & (wcus['In/Out/Thru'] == 'Inbound Receiving')
            & (wcus['VesselDraftFt'] >= min_draft_ft)
            & (wcus['TrafficCode'] == 1) # Domestic (Trips & Drafts)
            ,
        :
    ]
# wcus_all_in.head()

In [171]:
inports.merge(wcus, how='left')['WaterwayName'].unique()

array(['Indiana Harbor, IN', 'Gary, IN', 'Burns Waterway Harbor, IN',
       'Toledo-Lucas County Port, OH', 'Cleveland-Cuyahoga Port, OH',
       'Rouge River, MI', 'Conneaut Harbor, OH',
       'Ashtabula Port Authority, OH',
       'Calumet Harbor and River, IL and IN', 'Chicago Harbor, IL'],
      dtype=object)

In [172]:
trips_burns = wcus_all_in.loc[
    wcus_all_in['WaterwayName'] == 'Burns Waterway Harbor, IN',
    cols_keep
]

In [179]:
trip_counts_burns = trips_burns.groupby(['WaterwayCode', 'WaterwayName', 'In/Out/Thru', 'VesselTypeName', 'CompletedYear']).sum().reset_index()


In [ ]:
# trip_counts_burns.drop(columns=['VesselDraftFt']).to_csv('../data/clean/trip_counts/inbound/burns.csv')

### Soo Locks through-trips

In [181]:
wcus.loc[
        wcus['WaterwayName'].str.startswith('Sault') & (wcus['VesselTypeName']=='Self-Propelled dry')& (wcus['VesselDraftFt']>=12), 
        :
    ].head(3)

,RegionName,WaterwayCode,WaterwayName,TrafficCode,TrafficName,In/Out/Thru,Up/Down,VesselType,VesselTypeName,VesselDraftFt,Trips,CompletedYear
188831,GREAT LAKES,3817,"Sault Ste Marie, MI",1,Domestic (Trips & Drafts),Inbound Receiving,Port,1.0,Self-Propelled dry,12,1,2021
188832,GREAT LAKES,3817,"Sault Ste Marie, MI",1,Domestic (Trips & Drafts),Inbound Receiving,Port,1.0,Self-Propelled dry,15,1,2021
188833,GREAT LAKES,3817,"Sault Ste Marie, MI",1,Domestic (Trips & Drafts),Inbound Receiving,Port,1.0,Self-Propelled dry,16,1,2021


In [83]:
# wcus['Up/Down'].unique()
# wcus.loc[ (wcus['In/Out/Thru']!='Waterway'), :]

In [84]:
# wcus_all.loc[
#         (wcus_all['CompletedYear']==2022) & (wcus_all['WaterwayName']=='Duluth-Superior, MN and WI'),
#         :
#     ]

## WCUS Cargo cleaning steps

1. Filter to only WaterwayCodes in `inports` and `outports`
2. `CommodityCode` 4410 (iron ore)
3. Only Lakewise vessel movements (no intraport movements, for example)

## WCUS Cargo cleaning

In [182]:
path = "../data/raw/Cargo_AllRegions_10yr_2013-2022.xlsx"

cargo = pd.read_excel(path, sheet_name="Cargo_AllRegions_10yr_2013-2022")

In [254]:
cols_to_drop_cargo = ['TrafficCode', 'Allo1Code', 'Allo2Code', 'Up/Down', 'TonMiles']

In [255]:
for code, port in outports_dict.items():
    
    cargoes_port = cargo.loc[
            (cargo['WaterwayCode']==code)
            & (cargo['CommodityCode']==4410)
            & (cargo['TrafficName']=='Lakewise')
            & (cargo['In/Out/Thru']=='Outbound Shipping'), 
            :
        ].drop(columns = cols_to_drop_cargo)


    cargoes_port.to_csv(
            '../data/clean/cargoes/outbound/{}.csv'.format(port),
            index=False
        )


In [232]:
cargo.loc[(cargo['WaterwayCode']==3619) & (cargo['CommodityName']=='Iron Ore'), :]

,WaterwayCode,WaterwayName,TrafficCode,TrafficName,CommodityCode,CommodityName,Allo1Code,In/Out/Thru,Allo2Code,Up/Down,ShortTons,TonMiles,CompletedYear
257102,3619,"Presque Isle Township, MI",22,Canadian Exports,4410,Iron Ore,2,Outbound Shipping,0,Port,21076,0,2015
257103,3619,"Presque Isle Township, MI",22,Canadian Exports,4410,Iron Ore,2,Outbound Shipping,0,Port,17000,0,2016
257104,3619,"Presque Isle Township, MI",22,Canadian Exports,4410,Iron Ore,2,Outbound Shipping,0,Port,22,0,2017


This explains that Presque Isle only exports to Canada! No domestic shipments of iron ore from here.

In [257]:
outports_dict

{3924: 'duluth_superior',
 3926: 'two_harbors',
 3845: 'presque_isle',
 3841: 'marquette_harbor'}

In [258]:
# test crosswalk
cw = pd.read_csv('../data/crosswalks/port_id_to_waterwaycode_outbound.csv')

In [259]:
cw

,WaterwayCode,OBJECTID,shortname
0,3924,3,duluth_superior
1,3841,270,marquette_harbor
2,3926,35,two_harbors
3,3845,270,presque_isle
